In [65]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough

In [92]:
# Creates the YouTube transcript API and specifies the video. Gfr50f6ZBvo
video_id = "Gfr50f6ZBvo"

ytt_api = YouTubeTranscriptApi()

In [93]:
# Get English Transcript

transcript_list = ytt_api.list(video_id)

transcript = None

# First preference: English transcript
for t in transcript_list:
    if t.language_code == "en":
        transcript = t.fetch()
        break

# If English isn't available, translate an available transcript to English
if transcript is None:
    for t in transcript_list:
        if t.is_translatable:
            transcript = t.translate("en").fetch()
            break

if transcript is None:
    raise ValueError("No English or translatable transcript available.")

print("Transcript loaded successfully.")

Transcript loaded successfully.


In [75]:
# print transcript
for snippet in transcript:
    print(snippet.text)

the following is a conversation with
demus hasabis
ceo and co-founder of deepmind
a company that has published and builds
some of the most incredible artificial
intelligence systems in the history of
computing including alfred zero that
learned
all by itself to play the game of gold
better than any human in the world and
alpha fold two that solved protein
folding
both tasks considered nearly impossible
for a very long time
demus is widely considered to be one of
the most brilliant and impactful humans
in the history of artificial
intelligence and science and engineering
in general
this was truly an honor and a pleasure
for me to finally sit down with him for
this conversation and i'm sure we will
talk many times again in the future
this is the lex friedman podcast to
support it please check out our sponsors
in the description and now dear friends
here's demis
hassabis
let's start with a bit of a personal
question
am i an ai program you wrote to
interview people until i get good enough


In [43]:
print(transcript[0])

FetchedTranscriptSnippet(text='the following is a conversation with', start=0.08, duration=3.44)


In [77]:
# Combines all transcript snippets into one text string
transcript_text = " ".join(
    snippet.text for snippet in transcript
)

print(transcript_text[:1000])

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [ ]:
# Convert this transcript into a LangChain Document object
document = Document(
    page_content=transcript_text,
    metadata={
        "source": "youtube",
        "video_id": video_id
    }
)
print(document)

page_content='the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i ge

In [81]:
# Breaks the long transcript into smaller overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents([document])

print(f"Total chunks: {len(chunks)}")

Total chunks: 168


In [82]:
# Create embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3340.31it/s]


In [84]:
# Create FAISS (Stores the chunk embeddings in FAISS so we can perform similarity search)
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(type(vector_store))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [85]:
# Create the Retriever (retrieves the 3 most relevant chunks)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [86]:
# Create ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [87]:
# Create the prompt (Telling the LLM to answer using the retrieved transcript context)
prompt = ChatPromptTemplate.from_template("""
You are a helpful YouTube video assistant.

Answer the user's question using only the provided context from the video transcript.

If the answer cannot be found in the context, say:
"I couldn't find the answer in the video transcript."

Context:
{context}

Question:
{question}
""")

In [88]:
# 1st Way ( manually )
query = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed in short tell me"

retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

formatted_prompt = prompt.invoke({
    "context": context,
    "question": query
})

response = llm.invoke(formatted_prompt)

print(response.content)

Yes. The video discusses nuclear fusion. It describes how the speakers have developed a controller that can shape, contain, and hold plasma for long periods, mentioning specific plasma shapes (like droplets) that improve energy production. They also talk about collaborating with fusion experts (e.g., EPFL) and using AI—particularly deep reinforcement learning—to control magnetic fields in tokamak reactors, aiming to tackle the physics, materials, and engineering challenges of fusion.


In [89]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    } | prompt | llm
)

response = rag_chain.invoke("can you summarize the video ?")
print(response.content)

The video is a conversational podcast that mixes humor, philosophical musings, and personal anecdotes. It opens with a sponsor plug and then the host asks a tongue‑in‑cheek question to guest Demis Hassabis: “Am I an AI program you wrote to interview people?” The discussion touches on whether an AI should be told it is an AI, likening it to a meta‑Turing test and a Heisenberg‑type effect where knowledge of the truth could change behavior. The hosts also reference a “benchmark from the future” that replays 2022 before AIs were strong enough, wondering if the program would pass.

Later the episode includes a short quote from Edskar Dykstra: “Computer science is no more about computers than astronomy is about telescopes,” followed by a “day‑in‑the‑life” segment. The hosts ask about habits such as sleep patterns, coffee consumption, computer setup (screens, keyboards, editors like Emacs/Vim), and compare how work used to be 10–20 years ago—focused research, programming, reading papers, and 